# GISLR — ASL-LEX parameter classifiers (continuous features, held out by gloss)

**What this notebook does.** Training pipeline stage (not a diagnostic — this one fits real
models). Follow-up to `gislr.1.pipeline.stage1-rules.ipynb` (TODO §14): that notebook's rule
engine hand-picks a threshold per ASL-LEX 2.0 parameter (e.g. `spread = mean adjacent-finger
angle > 20°`); several parameters sit near chance (`flexion`, `spread`, `repeated_movement`,
`contact`, `marked_handshape`) with no clean threshold. This notebook trains one small classifier
per parameter on the **same underlying continuous measurement** the rule engine uses (not the
binarized code), to test whether the threshold — not the measurement — was the bottleneck.

**User's design choices, asked directly (this conversation, 2026-09-28):**
- **Input**: continuous pre-threshold measurements (`sb.recognize.recognition2.rules.continuous_features`
  — 23 scalars: finger extension/flexion/spread angles, thumb distances, rotation, movement
  path stats, location distances, two-handedness symmetry), not the rule engine's discrete codes.
- **Task**: 18 separate per-parameter classifiers (not one 233-way sign classifier).

**The held-out split is by gloss, not by clip.** A clip-level split would let a classifier learn
\"which of the 233 signs is this\" and get the ASL-LEX label for free from sign identity, since
every clip of a gloss shares the same label — that would test memorization, not whether the
continuous features actually carry the phonological signal. `TEST_GLOSS_FRAC` of the 233 mapped
glosses are held out entirely; every one of their clips is eval-only.

**Claude does not execute the training cells (§3 onward) in this notebook** — per this repo's
convention, model training is built and handed off, not run by Claude. §1–2 (feature loading, the
gloss split) are deterministic/no-training and were run once to sanity-check the pipeline.

**Artifacts:**

| path | what |
|---|---|
| `data/cache/gislr_aslex_rules/continuous_features.jsonl` | precomputed input (one row/clip, from `precompute_continuous_features.py`-equivalent logic in §1) |
| `data/cache/gislr_aslex_rules/classifier_results.csv` | held-out-gloss accuracy/balanced-accuracy per parameter, vs. the rule engine's agreement |
| `data/cache/gislr_aslex_rules/classifier_gloss_split.json` | which glosses were held out (for reproducibility) |

**Resumability**: §1 reads a manifest-driven cache (`continuous_features_manifest.json`) written by
the extraction step — safe to re-run, skips already-extracted clips. §3's training loop is cheap
(HistGradientBoostingClassifier, tens of thousands of rows, seconds per parameter) — no manifest
needed, just re-run in full.

## Setup

In [1]:
# ============================================================
# Imports, tunables, resolved paths
# ============================================================
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, gislr_dir
from sb.recognize.aslex import gloss_entries
from sb.recognize.recognition2.rules import CONTINUOUS_FEATURES, PARAMETERS

SEED = 0
TEST_GLOSS_FRAC = 0.20   # share of the 233 mapped glosses held out entirely (clip-level eval)
MIN_CLASS_COUNT = 4      # a parameter/class with fewer held-out clips than this is dropped from the table

OUT_DIR = CACHE_DIR / "gislr_aslex_rules"
FEATURES_PATH = OUT_DIR / "continuous_features.jsonl"

# per-parameter agreement already measured by the rule engine (stage1-rules.ipynb, full test
# split, no-pose) -- the baseline this notebook's classifiers are trying to beat.
RULE_ENGINE_AGREEMENT = {
    "ulnar_rotation": 0.847, "spread_change": 0.769, "thumb_contact": 0.681,
    "flexion_change": 0.679, "sign_type": 0.667, "thumb_position": 0.609,
    "major_location": 0.471, "movement": 0.477, "selected_fingers": 0.476,
    "minor_location": 0.267, "handshape": 0.110, "contact": 0.583,
    "repeated_movement": 0.543, "marked_handshape": 0.551, "spread": 0.523,
    "flexion": 0.252, "second_minor_location": 0.390, "non_dominant_handshape": 0.028,
}

print(f"data:     {gislr_dir()}")
print(f"features: {FEATURES_PATH}")
print(f"test gloss frac: {TEST_GLOSS_FRAC}, seed: {SEED}")

data:     C:\Users\user2\.cache\kagglehub\datasets\bracu23101281\gislr-stratified\versions\1
features: C:\Users\user2\repository\data\cache\gislr_aslex_rules\continuous_features.jsonl
test gloss frac: 0.2, seed: 0


## 1. Load precomputed continuous features

One row per clip: its gloss, which GISLR split it came from, its ASL-LEX ground truth per
parameter (`None` where the gloss doesn't map or its variants disagree — same policy as
`aslex.gloss_codes`), and the 23 continuous measurements from `rules.continuous_features`.
No training here — reads the cache written by the extraction step.

In [2]:
# ============================================================
# Load the feature cache into a flat DataFrame
# ============================================================
records = []
with open(FEATURES_PATH, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        row = {"uid": rec["uid"], "gloss": rec["gloss"], "split": rec["split"]}
        row.update({f"truth__{k}": v for k, v in rec["truth"].items()})
        row.update(rec["features"])
        records.append(row)

df = pd.DataFrame(records)
print(f"{len(df)} clips, {df['gloss'].nunique()} glosses")
df[list(CONTINUOUS_FEATURES)].describe().T[["count", "mean", "std", "min", "max"]]

46600 clips, 233 glosses


,count,mean,std,min,max
ext_thumb,39016.0,0.962974,0.044878,0.538390,0.999799
ext_index,39016.0,0.902536,0.156854,0.073683,0.999980
ext_middle,39016.0,0.808466,0.225620,0.055606,0.999982
ext_ring,39016.0,0.753350,0.248575,0.021668,0.999971
ext_pinky,39016.0,0.795017,0.230667,0.059582,0.999914
base_flex,39016.0,42.119879,23.683110,3.856480,170.562027
nonbase_flex,39016.0,69.369304,48.380219,3.406182,178.955719
spread_angle,39016.0,23.449421,11.872025,1.794622,97.630585
thumb_open,39016.0,0.702340,0.248383,0.033007,3.141462
thumb_contact_dist,39016.0,0.648189,0.366557,0.015286,5.663939


## 2. Hold out glosses (not clips)

A fixed, seeded random `TEST_GLOSS_FRAC` of the mapped glosses are held out entirely — every clip
of a held-out gloss goes to eval only, and no held-out gloss's clips are seen during training for
any parameter. Deterministic, no training.

In [3]:
# ============================================================
# Gloss-level train/test split
# ============================================================
all_glosses = sorted(df["gloss"].unique())
rng = np.random.default_rng(SEED)
n_test = max(1, int(round(len(all_glosses) * TEST_GLOSS_FRAC)))
test_glosses = set(rng.choice(all_glosses, size=n_test, replace=False))
train_glosses = set(all_glosses) - test_glosses

train_df = df[df["gloss"].isin(train_glosses)].reset_index(drop=True)
test_df = df[df["gloss"].isin(test_glosses)].reset_index(drop=True)

(OUT_DIR / "classifier_gloss_split.json").write_text(
    json.dumps({"seed": SEED, "test_glosses": sorted(test_glosses)}, indent=1), encoding="utf-8")
print(f"train: {len(train_glosses)} glosses / {len(train_df)} clips")
print(f"test:  {len(test_glosses)} glosses / {len(test_df)} clips (held out, never seen in training)")

train: 186 glosses / 37200 clips
test:  47 glosses / 9400 clips (held out, never seen in training)


## 3. Train one classifier per parameter — run this yourself

**This cell fits real models — Claude did not run it.** One `HistGradientBoostingClassifier` per
ASL-LEX parameter (native NaN handling, since not every clip has every continuous feature —
e.g. a clip with the hand out of frame has no `ext_*`; the tree learns to route around missing
features rather than needing imputation). `class_weight="balanced"` because several parameters are
heavily skewed at the gloss level (`ulnar_rotation` ground truth is 1 in only ~30 of 219 mapped
glosses, per the mean-vs-truth chart earlier this conversation) — unweighted training would let the
majority class dominate the loss the same way it dominates raw accuracy.

All 23 `CONTINUOUS_FEATURES` go in for every parameter — the tree does its own feature selection,
so e.g. `flexion`'s classifier can still pick up a location feature if that turns out informative,
the same way ASL-LEX's own `Handshape` column is defined from five combined sub-features.

In [ ]:
# ============================================================
# Train + evaluate, one classifier per parameter
# ============================================================
FEATURE_COLS = list(CONTINUOUS_FEATURES)
results = []
models = {}

for param in PARAMETERS:
    col = f"truth__{param}"
    tr = train_df.dropna(subset=[col])
    te = test_df.dropna(subset=[col])
    if tr[col].nunique() < 2 or len(te) < MIN_CLASS_COUNT:
        results.append({"parameter": param, "skipped": True, "reason": "too few classes/held-out clips"})
        continue

    X_tr, y_tr = tr[FEATURE_COLS].to_numpy(dtype=float), tr[col].astype(str).to_numpy()
    X_te, y_te = te[FEATURE_COLS].to_numpy(dtype=float), te[col].astype(str).to_numpy()

    clf = HistGradientBoostingClassifier(class_weight="balanced", max_iter=200, random_state=SEED)
    clf.fit(X_tr, y_tr)
    models[param] = clf

    pred = clf.predict(X_te)
    majority = tr[col].value_counts().idxmax()
    results.append({
        "parameter": param,
        "skipped": False,
        "n_train_clips": len(tr), "n_train_glosses": tr["gloss"].nunique(),
        "n_test_clips": len(te), "n_test_glosses": te["gloss"].nunique(),
        "n_classes": tr[col].nunique(),
        "accuracy": accuracy_score(y_te, pred),
        "balanced_accuracy": balanced_accuracy_score(y_te, pred),
        "majority_baseline": accuracy_score(y_te, [majority] * len(y_te)),
        "rule_engine_agreement": RULE_ENGINE_AGREEMENT.get(param, float("nan")),
    })

results_df = pd.DataFrame(results)
results_df.to_csv(OUT_DIR / "classifier_results.csv", index=False)
results_df

c:\Users\user2\repository\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\user2\repository\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\user2\repository\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\user2\repository\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Users\user2\repository\.venv\Lib\site-packages\sklearn\metrics\_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
c:\Us

## 4. Reading the comparison

Three numbers to read together per row, not `accuracy` alone:

- **`accuracy` vs. `majority_baseline`**: a classifier that scores *below* its own majority
  baseline learned nothing — that baseline is what "always guess the most common class" gets for
  free, on the *same held-out glosses*.
- **`balanced_accuracy` vs. `accuracy`**: on a skewed parameter (e.g. `ulnar_rotation`), a high
  `accuracy` with a much lower `balanced_accuracy` means the classifier is mostly predicting the
  majority class well and the minority class poorly — the same failure mode the gloss-mean chart
  found in the rule engine's raw `ulnar_rotation` agreement number this conversation.
- **`accuracy` vs. `rule_engine_agreement`**: the direct answer to "did training beat
  threshold-picking?" — expected to help most on `flexion`/`spread`/`repeated_movement`/`contact`
  (near-chance in stage1, and the mean-vs-truth chart showed real overlap between ASL-LEX's two
  categories at the raw-measurement level for some of these, so a classifier may not fix all of
  them either) and expected to help least on `ulnar_rotation`/`spread_change` (already strong,
  little room to gain, and few held-out positive examples to learn from).

Note the comparison isn't perfectly apples-to-apples: stage1's `rule_engine_agreement` was measured
clip-by-clip against every mapped gloss (233, full canonical test split); this notebook's `accuracy`
is measured only on the `TEST_GLOSS_FRAC` held-out glosses (~47 of 233) — a harder, stricter test
(never-seen signs) but on fewer glosses, so read a close row as "roughly tied," not literally equal.